# Final Project: Neural Networks on Tabular Data

In [ ]:
#imports

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras import Input
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.models import load_model
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score



In [ ]:
#read in data
df = pd.read_csv("mental_health_data.csv")

## Phase 1: Data Analysis and Preparation

In [ ]:
#check
df.head(10)

### Exploratory Data Analysis

In [ ]:
#basic EDA
df.describe()

df.hist(figsize=(15,10))
plt.tight_layout()
plt.show()

In [ ]:
#both mood and stress level have no real correlation, ill just leave these out
print(df.groupby("diet_quality")["mood_score"].mean())
df.groupby("weather")["mood_score"].mean()

In [ ]:
#grab numerical columns
numerical_cols = df.select_dtypes(include=['float64', 'int64'])

#IQR method to find outliers
outlier_summary = []
for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((df[col] < (Q1 - 1.5 * IQR)) | (df[col] > (Q3 + 1.5 * IQR))).sum()
    outlier_pct = (outliers / len(df)) * 100
    outlier_summary.append({
        'Feature': col,
        'Outliers': outliers,
        'Percentage': f"{outlier_pct:.2f}%"
    })

outlier_df = pd.DataFrame(outlier_summary)
print(outlier_df.to_string(index=False))

#check if outliers are realistic, they are so i wont remove them
for col in numerical_cols:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]

    print(f"\n--- {col} ---")
    print(outliers[col].sort_values())

In [ ]:
#create thresholds for 3 classifications of mood
#chose thresholds based on unbalanced data so im deciding anything at 5 or below is 'sad'
def classify_mood(x):
    if x <= 5:
        return "Sad"
    elif x <= 7:
        return "Neutral"
    else:
        return "Happy"

df["mood_class"] = df["mood_score"].apply(classify_mood)

In [ ]:
#plot
df["mood_class"].value_counts().plot(kind="bar")

plt.title("Mood Class Distribution")
plt.xlabel("Mood Class")
plt.ylabel("Count")

plt.show()

In [ ]:
#create correlation matrix
correlation_matrix = numerical_cols.corr()

#heatmap of matrix
plt.figure(figsize=(12,8))

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Feature Correlation Heatmap")

plt.show()

In [ ]:
#after seeing more correlation with stress level im going to continue with that as my target

#create thresholds for 3 classifications of stress, mess around with thresholds to try and balance the counts
#lower stress is better, higher stress is worse

def classify_stress(x):
    if x <= 2:
        return "Low"
    elif x <= 4:
        return "Moderate"
    else:
        return "High"

df["stress_class"] = df["stress_level"].apply(classify_stress)

#plot class distribution
df["stress_class"].value_counts().plot(kind="bar")

plt.title("Stress Class Distribution")
plt.xlabel("Stress Class")
plt.ylabel("Count")

plt.show()

#### Manual Normalization

In [ ]:
#"no preprocessing shortcuts"

#copy to new normalized df
df_norm = df.copy()

#columns we want to normalize
cols_to_normalize = [
    "sleep_hours",
    "screen_time",
    "exercise_minutes",
    "daily_pending_tasks",
    "interruptions",
    "fatigue_level",
    "social_hours",
    "coffee_cups"
]

#manually normalize each column
for col in cols_to_normalize:

    min_value = df_norm[col].min()
    max_value = df_norm[col].max()

    df_norm[col] = (df_norm[col] - min_value) / (max_value - min_value)

df_norm[cols_to_normalize].describe()

## Phase 2: Prediction Without Machine Learning

### Predictor 1: Additive Scoring Predictor

$
S = b + w_1x_1 + w_2x_2 - w_3x_3
$

$
S = \text{predicted stress score}
$

$
b = \text{baseline stress}
$

$
x_i = \text{normalized feature values}
$

$
w_i = \text{feature weights / importance}
$

In [ ]:
def stress_predictor(row):

    # start at medium stress (lowered baseline from original)
    score = 0.4

    # increase stress(slight lower in task weight)
    score += 0.5 * row["daily_pending_tasks"]
    score += 0.2 * row["interruptions"]

    # decrease stress
    score -= 0.3 * row["sleep_hours"]

    return score

#apply predictor on features
df_norm["stress_prediction_score"] = df_norm.apply(stress_predictor, axis=1)
df_norm[[
    "daily_pending_tasks",
    "interruptions",
    "sleep_hours",
    "stress_prediction_score"
]].head(5)

#### Accuracy test with thresholds

In [ ]:
#convert prediction scores into stress classes

def predicted_stress_class(score):
    #upped "low" score
    if score < 0.4:
        return "Low"

    elif score < 0.66:
        return "Moderate"

    else:
        return "High"


#create predicted class column
df_norm["predicted_stress_class"] = df_norm["stress_prediction_score"].apply(predicted_stress_class)


#see actual vs predicted
print(df_norm[[
    "stress_class",
    "predicted_stress_class"
]].head())

#count correct predictions
correct_predictions = (
    df_norm["stress_class"] ==
    df_norm["predicted_stress_class"]
).sum()


#total predictions
total_predictions = len(df_norm)


#accuracy formula
accuracy = correct_predictions / total_predictions


print("Accuracy:", accuracy)

Initial predictor performance was poor because the scoring system produced too many moderate/high stress predictions. By adjusting the baseline, decision thresholds and reducing the weight of pending tasks, the score distribution better aligned with the actual class distribution, increasing accuracy from 24.4% to 58.9%.

### Predictor 2: Rule-Based Threshold Predictor

In [ ]:
def threshold_stress_predictor(row):

    score = 0.39

    # individual rules
    if row["daily_pending_tasks"] > 0.6:
        score += 0.30

    if row["interruptions"] > 0.5:
        score += 0.20

    if row["sleep_hours"] < 0.4:
        score += 0.25

    # combination rule
    if row["daily_pending_tasks"] > 0.6 and row["sleep_hours"] < 0.4:
        score += 0.20

    # protective rule
    if row["daily_pending_tasks"] < 0.3 and row["sleep_hours"] > 0.6:
        score -= 0.20

    # keep score between 0 and 1
    score = max(0, min(1, score))

    return score

In [ ]:
df_norm["threshold_prediction_score"] = df_norm.apply(threshold_stress_predictor, axis=1)

#again convert to classes
def threshold_score_to_class(score):

    if score < 0.4:
        return "Low"
    elif score < 0.66:
        return "Moderate"
    else:
        return "High"

df_norm["threshold_predicted_class"] = df_norm["threshold_prediction_score"].apply(threshold_score_to_class)

#see actual vs predicted
print(df_norm[[
    "stress_class",
    "threshold_predicted_class"
]].head())

#count correct predictions
correct_predictions = (
    df_norm["stress_class"] ==
    df_norm["threshold_predicted_class"]
).sum()


#total predictions
total_predictions = len(df_norm)


#accuracy formula
accuracy = correct_predictions / total_predictions


print("Accuracy:", accuracy)

### Predictor 3: Hybrid Rule-Weighted Predictor

In [ ]:
#nowill try to merge the first two predictors, in hopes of achieving higher accuracy than both

def hybrid_stress_predictor(row):

    score = 0.39

    # weighted part
    score += 0.45 * row["daily_pending_tasks"]
    score += 0.15 * row["interruptions"]
    score -= 0.25 * row["sleep_hours"]

    # rule-based adjustments
    if row["daily_pending_tasks"] > 0.7:
        score += 0.10

    if row["sleep_hours"] < 0.35:
        score += 0.10

    if row["daily_pending_tasks"] > 0.6 and row["sleep_hours"] < 0.4:
        score += 0.15

    if row["daily_pending_tasks"] < 0.3 and row["sleep_hours"] > 0.6:
        score -= 0.15

    # keep between 0 and 1
    score = max(0, min(1, score))

    return score

In [ ]:
df_norm["hybrid_prediction_score"] = df_norm.apply(hybrid_stress_predictor, axis=1)

def hybrid_score_to_class(score):

    if score < 0.47:
        return "Low"
    elif score < 0.64:
        return "Moderate"
    else:
        return "High"

df_norm["hybrid_predicted_class"] = df_norm["hybrid_prediction_score"].apply(hybrid_score_to_class)

#see actual vs predicted
print(df_norm[[
    "stress_class",
    "hybrid_predicted_class"
]].head())

#count correct predictions
correct_predictions = (
    df_norm["stress_class"] ==
    df_norm["hybrid_predicted_class"]
).sum()


#total predictions
total_predictions = len(df_norm)


#accuracy formula
accuracy = correct_predictions / total_predictions


print("Accuracy:", accuracy)

| Predictor   | Type                 | Accuracy |
| ----------- | -------------------- | -------- |
| Predictor 1 | Weighted Additive    | 58.9%    |
| Predictor 2 | Rule-Based Threshold | 63.4%    |
| Predictor 3 | Hybrid Rule-Weighted | 80.0%    |


## Phase 3: Overfitting Experiment

#### Neural Network tries to learn function:

$
X \rightarrow y 
$

#### Meaning:

$
\text{lifestyle features} \rightarrow \text{stress class}
$

In [ ]:
#start by taking all needed features and creating the input matrix
feature_cols = [
    "sleep_hours",
    "screen_time",
    "exercise_minutes",
    "daily_pending_tasks",
    "interruptions",
    "fatigue_level",
    "social_hours",
    "coffee_cups"
]

X = df_norm[feature_cols]

#target
y = df_norm["stress_class"]

In [ ]:
# manually encode stress classes into numbers

stress_mapping = {
    "Low": 0,
    "Moderate": 1,
    "High": 2
}

#replaces each word with its assigned number
y_encoded = y.map(stress_mapping)

print(y_encoded.head())

In [ ]:
# manually one-hot encode the target classes

y_onehot = []

for label in y_encoded:

    if label == 0:
        y_onehot.append([1, 0, 0])

    elif label == 1:
        y_onehot.append([0, 1, 0])

    else:
        y_onehot.append([0, 0, 1])

In [ ]:
# create a neural network model

model1 = Sequential()

# use 3 output neurons for Low, Moderate, and High stress
# softmax converts outputs into class probabilities
model1.add(
    Dense(
        3,
        activation="softmax",
        input_shape=(X.shape[1],)
    )
)

In [ ]:
#compile model, optimizer updates weights,cross entropy to minimize loss
model1.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
X_array = X.to_numpy().astype("float32")
y_array = np.array(y_onehot).astype("float32")

history1 = model1.fit(
    X_array,
    y_array,
    epochs=200,
    batch_size=32,
    verbose=1
)

In [ ]:
model2 = Sequential()

model2.add(Input(shape=(X_array.shape[1],)))

model2.add(Dense(8, activation="relu"))

model2.add(Dense(3, activation="softmax"))

model2.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history2 = model2.fit(
    X_array,
    y_array,
    epochs=70,
    batch_size=32,
    verbose=1
)

In [ ]:
model3 = Sequential()

model3.add(Input(shape=(X_array.shape[1],)))

model3.add(Dense(32, activation="relu"))
model3.add(Dense(16, activation="relu"))

model3.add(Dense(3, activation="softmax"))

model3.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history3 = model3.fit(
    X_array,
    y_array,
    epochs=200,
    batch_size=32,
    verbose=1
)

In [ ]:
model4 = Sequential()

model4.add(Input(shape=(X_array.shape[1],)))

model4.add(Dense(128, activation="relu"))
model4.add(Dense(64, activation="relu"))
model4.add(Dense(32, activation="relu"))

model4.add(Dense(3, activation="softmax"))

model4.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history4 = model4.fit(
    X_array,
    y_array,
    epochs=500,
    batch_size=16,
    verbose=1
)

| Model   | Architecture  | Max Training Accuracy |
| ------- | ------------- | --------------------: |
| Model 1 | Output only   |                  ~89% |
| Model 2 | 8 neurons     |                  ~90% |
| Model 3 | 32 → 16       |                  ~92% |
| Model 4 | 128 → 64 → 32 |                  100% |


In [ ]:
overfit_results = {
    "Model": ["Model 1", "Model 2", "Model 3", "Model 4"],
    "Architecture": [
        "Output layer only",
        "8 hidden neurons",
        "32 → 16 hidden neurons",
        "128 → 64 → 32 hidden neurons"
    ],
    "Max Training Accuracy": [0.89, 0.90, 0.92, 1.00]
}

overfit_results_df = pd.DataFrame(overfit_results)

overfit_results_df

In [ ]:
plt.plot(history4.history["accuracy"])

plt.title("Model 4 Training Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Training Accuracy")

plt.show()

The purpose of this phase was to determine how large a neural network needed to be before it could overfit the full dataset. The simplest model, containing only a softmax output layer, plateaued around 89% training accuracy. Adding a small hidden layer improved performance slightly, and a two-layer model reached around 92%. The larger model with hidden layers of 128, 64, and 32 neurons reached 100% training accuracy, showing that this architecture was large enough to memorize the dataset. This model should be treated as an overfitting upper bound rather than the final model for generalization.

## Phase 4: Model Selection and Evaluation

#### Shuffle and Split

In [ ]:
# shuffle the dataset rows
shuffled_df = df_norm.sample(frac=1, random_state=42).reset_index(drop=True)

# split (80% training, 20% validation)
split_index = int(len(shuffled_df) * 0.8)

# create training set
train_df = shuffled_df[:split_index]

# create validation set
val_df = shuffled_df[split_index:]

# check sizes
print("Training rows:", len(train_df))
print("Validation rows:", len(val_df))

In [ ]:
#split training and validation into features and class
X_train = train_df[feature_cols]
y_train = train_df["stress_class"]


X_val = val_df[feature_cols]
y_val = val_df["stress_class"]

In [ ]:
#encode training targets
y_train_encoded = y_train.map(stress_mapping)

#encode validation targets
y_val_encoded = y_val.map(stress_mapping)

# one-hot encode targets
y_train_onehot = []

for label in y_train_encoded:

    if label == 0:
        y_train_onehot.append([1, 0, 0])

    elif label == 1:
        y_train_onehot.append([0, 1, 0])

    else:
        y_train_onehot.append([0, 0, 1])


y_val_onehot = []

for label in y_val_encoded:

    if label == 0:
        y_val_onehot.append([1, 0, 0])

    elif label == 1:
        y_val_onehot.append([0, 1, 0])

    else:
        y_val_onehot.append([0, 0, 1])

#transform data to tensorFlow friendly arrays
X_train_array = X_train.to_numpy().astype("float32")
X_val_array = X_val.to_numpy().astype("float32")

y_train_array = np.array(y_train_onehot).astype("float32")
y_val_array = np.array(y_val_onehot).astype("float32")

#### Model 1

In [ ]:
#reuse inital small model with model checkpointing
checkpoint1 = ModelCheckpoint(
    "model1_best.keras",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

p4_history1 = model1.fit(
    X_train_array,
    y_train_array,
    epochs=200,
    batch_size=32,
    validation_data=(X_val_array, y_val_array),
    callbacks=[checkpoint1],
    verbose=1
)

In [ ]:
# load best checkpointed model
best_model1 = load_model("model1_best.keras")

# generate prediction probabilities
predictions = best_model1.predict(X_val_array)

# convert probabilities into predicted class indices
predicted_classes = np.argmax(predictions, axis=1)

# convert one-hot encoded validation targets back into class indices
true_classes = np.argmax(y_val_array, axis=1)

# calculate evaluation metrics
accuracy = accuracy_score(true_classes, predicted_classes)

precision = precision_score(
    true_classes,
    predicted_classes,
    average="weighted"
)

recall = recall_score(
    true_classes,
    predicted_classes,
    average="weighted"
)

f1 = f1_score(
    true_classes,
    predicted_classes,
    average="weighted"
)

# print results
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

In [ ]:
plt.plot(p4_history1.history["accuracy"], label="Training Accuracy")
plt.plot(p4_history1.history["val_accuracy"], label="Validation Accuracy")

plt.title("Model 1 Learning Curve")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.show()

#### Model 2

In [ ]:
checkpoint2 = ModelCheckpoint(
    "model2_best.keras",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

model2 = Sequential()

model2.add(Input(shape=(X_train_array.shape[1],)))
model2.add(Dense(8, activation="relu"))
model2.add(Dense(3, activation="softmax"))

model2.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

p4_history2 = model2.fit(
    X_train_array,
    y_train_array,
    epochs=200,
    batch_size=32,
    validation_data=(X_val_array, y_val_array),
    callbacks=[checkpoint2],
    verbose=1
)

In [ ]:
# load best checkpointed model
best_model2 = load_model("model2_best.keras")

# generate prediction probabilities
predictions = best_model2.predict(X_val_array)

# convert probabilities into predicted class indices
predicted_classes = np.argmax(predictions, axis=1)

# convert one-hot encoded validation targets back into class indices
true_classes = np.argmax(y_val_array, axis=1)

# calculate evaluation metrics
accuracy = accuracy_score(true_classes, predicted_classes)

precision = precision_score(
    true_classes,
    predicted_classes,
    average="weighted"
)

recall = recall_score(
    true_classes,
    predicted_classes,
    average="weighted"
)

f1 = f1_score(
    true_classes,
    predicted_classes,
    average="weighted"
)

# print results
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

In [ ]:
plt.plot(p4_history2.history["accuracy"], label="Training Accuracy")
plt.plot(p4_history2.history["val_accuracy"], label="Validation Accuracy")

plt.title("Model 2 Learning Curve")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.show()

In [ ]:
checkpoint3 = ModelCheckpoint(
    "model3_best.keras",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

model3 = Sequential()

model3.add(Input(shape=(X_train_array.shape[1],)))
model3.add(Dense(32, activation="relu"))
model3.add(Dense(16, activation="relu"))
model3.add(Dense(3, activation="softmax"))

model3.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

p4_history3 = model3.fit(
    X_train_array,
    y_train_array,
    epochs=200,
    batch_size=32,
    validation_data=(X_val_array, y_val_array),
    callbacks=[checkpoint3],
    verbose=1
)

In [ ]:
# load best checkpointed model
best_model3 = load_model("model3_best.keras")

# generate prediction probabilities
predictions = best_model3.predict(X_val_array)

# convert probabilities into predicted class indices
predicted_classes = np.argmax(predictions, axis=1)

# convert one-hot encoded validation targets back into class indices
true_classes = np.argmax(y_val_array, axis=1)

# calculate evaluation metrics
accuracy = accuracy_score(true_classes, predicted_classes)

precision = precision_score(
    true_classes,
    predicted_classes,
    average="weighted"
)

recall = recall_score(
    true_classes,
    predicted_classes,
    average="weighted"
)

f1 = f1_score(
    true_classes,
    predicted_classes,
    average="weighted"
)

# print results
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

In [ ]:
plt.plot(p4_history3.history["accuracy"], label="Training Accuracy")
plt.plot(p4_history3.history["val_accuracy"], label="Validation Accuracy")

plt.title("Model 3 Learning Curve")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.show()

In [ ]:
checkpoint4 = ModelCheckpoint(
    "model4_best.keras",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

model4 = Sequential()

model4.add(Input(shape=(X_train_array.shape[1],)))

model4.add(Dense(128, activation="relu"))
model4.add(Dense(64, activation="relu"))
model4.add(Dense(32, activation="relu"))

model4.add(Dense(3, activation="softmax"))

model4.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

p4_history4 = model4.fit(
    X_train_array,
    y_train_array,
    epochs=200,
    batch_size=32,
    validation_data=(X_val_array, y_val_array),
    callbacks=[checkpoint4],
    verbose=1
)

In [ ]:
# load best checkpointed model
best_model4 = load_model("model4_best.keras")

# generate prediction probabilities
predictions = best_model4.predict(X_val_array)

# convert probabilities into predicted class indices
predicted_classes = np.argmax(predictions, axis=1)

# convert one-hot encoded validation targets back into class indices
true_classes = np.argmax(y_val_array, axis=1)

# calculate evaluation metrics
accuracy = accuracy_score(true_classes, predicted_classes)

precision = precision_score(
    true_classes,
    predicted_classes,
    average="weighted"
)

recall = recall_score(
    true_classes,
    predicted_classes,
    average="weighted"
)

f1 = f1_score(
    true_classes,
    predicted_classes,
    average="weighted"
)

# print results
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

In [ ]:
plt.plot(p4_history4.history["accuracy"], label="Training Accuracy")
plt.plot(p4_history4.history["val_accuracy"], label="Validation Accuracy")

plt.title("Model 4 Learning Curve")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.show()

Model 2 ended up being the best overall balance between performance and complexity. It achieved the highest validation accuracy at around 92.25% while still using a relatively small architecture with only one hidden layer of 8 neurons. Unlike the larger models, it showed very little sign of overfitting since the training and validation curves stayed close together throughout training. The larger networks were able to learn more complex patterns, but they did not improve validation performance and eventually began memorizing the training data instead of generalizing as well to unseen data.

## Phase 5: Feature Importance and Reduction

#### For loop to train models with one feature at a time

In [ ]:
feature_importance_results = []

for feature in feature_cols:

    print("Training model using only:", feature)

    # use only one feature at a time
    X_train_single = train_df[[feature]].to_numpy().astype("float32")
    X_val_single = val_df[[feature]].to_numpy().astype("float32")

    # checkpoint for this feature
    checkpoint = ModelCheckpoint(
        f"{feature}_best.keras",
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=0
    )

    # same Model 2 structure, but input shape is only 1 feature
    single_feature_model = Sequential()

    single_feature_model.add(Input(shape=(1,)))
    single_feature_model.add(Dense(8, activation="relu"))
    single_feature_model.add(Dense(3, activation="softmax"))

    single_feature_model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    history = single_feature_model.fit(
        X_train_single,
        y_train_array,
        epochs=100,
        batch_size=32,
        validation_data=(X_val_single, y_val_array),
        callbacks=[checkpoint],
        verbose=0
    )

    # best validation accuracy for this feature
    best_val_accuracy = max(history.history["val_accuracy"])

    feature_importance_results.append({
        "Feature": feature,
        "Validation Accuracy": best_val_accuracy
    })

# turn results into dataframe
feature_importance_df = pd.DataFrame(feature_importance_results)

# sort best to worst
feature_importance_df = feature_importance_df.sort_values(
    by="Validation Accuracy",
    ascending=False
)

feature_importance_df

####  now iteratively remove the least important features and retrain to see how accuracy changes

In [ ]:
feature_sets_results = []

# ranked from weakest to strongest based on previous results
ranked_features = [
    "coffee_cups",
    "social_hours",
    "fatigue_level",
    "screen_time",
    "exercise_minutes",
    "sleep_hours",
    "interruptions",
    "daily_pending_tasks"
]

# start with all features
current_features = ranked_features.copy()

while len(current_features) > 0:

    print("Training with features:", current_features)

    # build train/validation inputs
    X_train_reduced = train_df[current_features].to_numpy().astype("float32")
    X_val_reduced = val_df[current_features].to_numpy().astype("float32")

    # build model
    reduced_model = Sequential()

    reduced_model.add(Input(shape=(len(current_features),)))
    reduced_model.add(Dense(8, activation="relu"))
    reduced_model.add(Dense(3, activation="softmax"))

    reduced_model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    # train model
    history = reduced_model.fit(
        X_train_reduced,
        y_train_array,
        epochs=100,
        batch_size=32,
        validation_data=(X_val_reduced, y_val_array),
        verbose=0
    )

    # record best validation accuracy
    best_val_accuracy = max(history.history["val_accuracy"])

    feature_sets_results.append({
        "Number of Features": len(current_features),
        "Features Used": current_features.copy(),
        "Best Validation Accuracy": best_val_accuracy
    })

    # remove weakest remaining feature
    current_features.pop(0)

# results dataframe
feature_reduction_df = pd.DataFrame(feature_sets_results)

feature_reduction_df

Although daily_pending_tasks was by far the strongest standalone predictor, validation accuracy improved substantially when additional features were included. This suggests that many weaker features still contributed useful complementary information through feature interactions learned by the neural network. However, removing lower-ranked features caused only minor decreases in performance, indicating that some variables contributed relatively little to overall prediction quality.

In [ ]:
phase4_results = pd.DataFrame({
    "Model": [
        "Model 1",
        "Model 2",
        "Model 3",
        "Model 4"
    ],

    "Architecture": [
        "Output Only",
        "8 Hidden Neurons",
        "32 → 16 Hidden Neurons",
        "128 → 64 → 32 Hidden Neurons"
    ],

    "Validation Accuracy": [
        0.8975,
        0.9225,
        0.9200,
        0.9175
    ],

    "Precision": [
        0.8974,
        0.9219,
        0.9196,
        0.9181
    ],

    "Recall": [
        0.8975,
        0.9225,
        0.9200,
        0.9175
    ],

    "F1 Score": [
        0.8942,
        0.9215,
        0.9192,
        0.9167
    ]
})

phase4_results

In [ ]:
plt.figure(figsize=(10,5))

plt.bar(
    feature_importance_df["Feature"],
    feature_importance_df["Validation Accuracy"]
)

plt.title("Single Feature Validation Accuracy")
plt.xlabel("Feature")
plt.ylabel("Validation Accuracy")

# zoom y-axis to show differences more clearly
plt.ylim(0.55, 0.80)

plt.xticks(rotation=45)

plt.show()

In [ ]:
plt.plot(
    feature_reduction_df["Number of Features"],
    feature_reduction_df["Best Validation Accuracy"],
    marker="o"
)

plt.title("Validation Accuracy vs Number of Features")
plt.xlabel("Number of Features")
plt.ylabel("Best Validation Accuracy")

plt.gca().invert_xaxis()

plt.show()